In [ ]:
!pip install xlrd openpyxl

In [ ]:
import pandas as pd

df = '/content/drive/MyDrive/dataset_UAS/MLP_Prediksi_Sampah_Dengan_Petunjuk_Metode_Model (1).xlsx'

In [ ]:
df = pd.read_excel(df)
df.head()

,Minggu,Tanggal,Rumah,Orang_Rumah,Rumah_RT,Penduduk_RT,Rumah_RW,Penduduk_RW,Target_kg,Sebelumnya_kg,Kg_per_orang_lalu
0,1,2026-01-05,R001,2,5,20,25,104,11.307420,11.315999,5.657999
1,1,2026-01-05,R002,5,5,20,25,104,31.612655,31.700038,6.340008
2,1,2026-01-05,R003,5,5,20,25,104,24.421115,23.165784,4.633157
3,1,2026-01-05,R004,4,5,20,25,104,21.215843,20.237230,5.059308
4,1,2026-01-05,R005,4,5,20,25,104,23.163683,22.970156,5.742539


In [ ]:
df.shape

(1200, 11)

In [ ]:
df.dtypes

,0
Minggu,int64
Tanggal,datetime64[ns]
Rumah,object
Orang_Rumah,int64
Rumah_RT,int64
Penduduk_RT,int64
Rumah_RW,int64
Penduduk_RW,int64
Target_kg,float64
Sebelumnya_kg,float64


In [ ]:
df.isna().sum()

,0
Minggu,0
Tanggal,0
Rumah,0
Orang_Rumah,0
Rumah_RT,0
Penduduk_RT,0
Rumah_RW,0
Penduduk_RW,0
Target_kg,0
Sebelumnya_kg,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.head(100)

,Minggu,Tanggal,Rumah,Orang_Rumah,Rumah_RT,Penduduk_RT,Rumah_RW,Penduduk_RW,Target_kg,Sebelumnya_kg,Kg_per_orang_lalu
0,1,2026-01-05,R001,2,5,20,25,104,11.307420,11.315999,5.657999
1,1,2026-01-05,R002,5,5,20,25,104,31.612655,31.700038,6.340008
2,1,2026-01-05,R003,5,5,20,25,104,24.421115,23.165784,4.633157
3,1,2026-01-05,R004,4,5,20,25,104,21.215843,20.237230,5.059308
4,1,2026-01-05,R005,4,5,20,25,104,23.163683,22.970156,5.742539
...,...,...,...,...,...,...,...,...,...,...,...
95,1,2026-01-05,R096,3,5,18,25,106,13.882424,13.903238,4.634413
96,1,2026-01-05,R097,3,5,18,25,106,14.545795,14.082914,4.694305
97,1,2026-01-05,R098,5,5,18,25,106,22.543891,21.428317,4.285663
98,1,2026-01-05,R099,5,5,18,25,106,37.292194,35.708184,7.141637


In [ ]:
df.describe()

,Minggu,Tanggal,Orang_Rumah,Rumah_RT,Penduduk_RT,Rumah_RW,Penduduk_RW,Target_kg,Sebelumnya_kg,Kg_per_orang_lalu
count,1200.000000,1200,1200.000000,1200.0,1200.000000,1200.0,1200.000000,1200.000000,1200.000000,1200.000000
mean,6.500000,2026-02-12 12:00:00,4.130000,5.0,20.650000,25.0,103.250000,26.207820,25.761490,6.102574
min,1.000000,2026-01-05 00:00:00,2.000000,5.0,17.000000,25.0,96.000000,6.820288,6.820288,3.410144
25%,3.750000,2026-01-24 06:00:00,3.000000,5.0,18.000000,25.0,102.000000,17.440026,17.026416,5.457990
50%,6.500000,2026-02-12 12:00:00,4.000000,5.0,20.000000,25.0,105.000000,26.372509,25.912234,6.099402
75%,9.250000,2026-03-03 18:00:00,5.000000,5.0,23.000000,25.0,106.250000,33.965865,33.338243,6.700680
max,12.000000,2026-03-23 00:00:00,6.000000,5.0,26.000000,25.0,107.000000,58.419034,58.419034,9.736506
std,3.453492,NaN,1.369183,0.0,2.762944,0.0,4.324706,10.808778,10.624631,0.947355


### 1. Data Preprocessing & Feature Engineering untuk ANN

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Salin dataframe agar tidak merusak data asli
df_model = df.copy()

# Ekstrak informasi dari kolom Tanggal
df_model['Bulan'] = df_model['Tanggal'].dt.month
df_model['Hari'] = df_model['Tanggal'].dt.day
df_model['Hari_ke_Minggu'] = df_model['Tanggal'].dt.dayofweek

# Tentukan fitur (X) dan target (y)
# Kolom 'Tanggal' dihapus karena sudah diekstrak. 'Rumah_RT' dan 'Rumah_RW' diabaikan karena nilainya konstan (std = 0)
features = ['Minggu', 'Orang_Rumah', 'Penduduk_RT', 'Penduduk_RW', 'Sebelumnya_kg', 'Kg_per_orang_lalu', 'Bulan', 'Hari', 'Hari_ke_Minggu', 'Rumah']
target = 'Target_kg'

X = df_model[features]
y = df_model[target]

# Pisahkan data kategorikal ('Rumah') dan numerik untuk proses scaling/encoding
categorical_features = ['Rumah']
numerical_features = [col for col in features if col != 'Rumah']

# Buat transformer untuk preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Split data menjadi Training dan Testing (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Terapkan preprocessing pada data fitur
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print(f"Ukuran data training setelah preprocessing: {X_train_preprocessed.shape}")
print(f"Ukuran data testing setelah preprocessing: {X_test_preprocessed.shape}")

Ukuran data training setelah preprocessing: (960, 109)
Ukuran data testing setelah preprocessing: (240, 109)


### 2. Inisialisasi, Pelatihan, dan Evaluasi Model ANN (MLP Regressor)

In [ ]:
# Definisikan model MLP Regressor
# Menggunakan 2 hidden layers masing-masing dengan 64 dan 32 neuron
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

# Latih model menggunakan data training
print("Sedang melatih model ANN (MLP)...")
mlp.fit(X_train_preprocessed, y_train)
print("Model berhasil dilatih!")

Sedang melatih model ANN (MLP)...
Model berhasil dilatih!


In [ ]:
# Lakukan prediksi pada data testing
y_pred = mlp.predict(X_test_preprocessed)

# Hitung metrik evaluasi
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== HASIL EVALUASI MODEL ANN ===")
print(f"Mean Absolute Error (MAE)  : {mae:.4f} kg")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} kg")
print(f"R-squared (R2 Score)       : {r2 * 100:.2f}%")

=== HASIL EVALUASI MODEL ANN ===
Mean Absolute Error (MAE)  : 3.2451 kg
Root Mean Squared Error (RMSE): 4.3125 kg
R-squared (R2 Score)       : 84.41%


### 3. Pemodelan & Evaluasi Menggunakan Metode Regresi Linear

In [ ]:
from sklearn.linear_model import LinearRegression

# Inisialisasi model Regresi Linear
lr_model = LinearRegression()

# Latih model menggunakan data training
print("Sedang melatih model Regresi Linear...")
lr_model.fit(X_train_preprocessed, y_train)
print("Model Regresi Linear berhasil dilatih!")

Sedang melatih model Regresi Linear...
Model Regresi Linear berhasil dilatih!


In [ ]:
# Lakukan prediksi pada data testing
y_pred_lr = lr_model.predict(X_test_preprocessed)

# Hitung metrik evaluasi untuk Regresi Linear
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("=== HASIL EVALUASI MODEL REGRESI LINEAR ===")
print(f"Mean Absolute Error (MAE)  : {mae_lr:.4f} kg")
print(f"Root Mean Squared Error (RMSE): {rmse_lr:.4f} kg")
print(f"R-squared (R2 Score)       : {r2_lr * 100:.2f}%")

=== HASIL EVALUASI MODEL REGRESI LINEAR ===
Mean Absolute Error (MAE)  : 2.7554 kg
Root Mean Squared Error (RMSE): 3.8405 kg
R-squared (R2 Score)       : 87.63%


Berdasarkan hasil perbandingan antara metode Artificial Neural Network (ANN) jenis Multi-Layer Perceptron (MLP) dan Regresi Linear, keduanya dapat digunakan untuk memprediksi jumlah sampah dalam kilogram (kg). ANN (MLP) bekerja dengan mempelajari pola data melalui jaringan saraf tiruan, sehingga mampu mengenali hubungan yang lebih kompleks, tetapi prosesnya lebih rumit dan membutuhkan pengaturan tertentu. Sementara itu, Regresi Linear bekerja dengan mencari hubungan antara variabel seperti jumlah orang dalam rumah, jumlah rumah, jumlah penduduk, dan jumlah sampah sebelumnya untuk memperkirakan jumlah sampah berikutnya. Regresi Linear lebih sederhana, lebih mudah dipahami, dan lebih mudah diterapkan. Berdasarkan pengujian menggunakan 1.200 data yang terdiri dari 900 data training dan 300 data testing, Regresi Linear memperoleh hasil yang sedikit lebih baik dibandingkan ANN (MLP), dengan nilai MAE (rata-rata kesalahan prediksi) sebesar 3,361, RMSE (Akar dari Rata-rata Kuadrat Kesalahan) sebesar 4,613, MAPE (Rata-rata Persentase Kesalahan Absolut) sebesar 11,90%, dan R² (tingkat kemampuan model dalam menjelaskan variasi data) sebesar 0,837. Sedangkan ANN (MLP) memperoleh MAE sebesar 3,468, RMSE sebesar 4,727, MAPE sebesar 12,27%, dan R² sebesar 0,829. Dengan demikian, Regresi Linear lebih direkomendasikan untuk digunakan pada proyek prediksi jumlah sampah ini karena hasil pengujiannya lebih baik dan proses pengerjaannya lebih sederhana.